# TMAP Quickstart Guide

This notebook walks through the complete TMAP pipeline:

1. **Encode** your data with MinHash
2. **Index** with LSHForest
3. **Layout** in 2D with force-directed algorithm
4. **Visualize** as interactive HTML

By the end, you'll have a working TMAP visualization!

## Setup

First, let's import the necessary modules and create some sample data.

In [1]:
import numpy as np

# TMAP imports
from tmap import LSHForest, MinHash
from tmap.layout import LayoutConfig, layout_from_lsh_forest
from tmap.visualization import TmapViz

print("TMAP imported successfully!")


TMAP imported successfully!


## Step 0: Create Sample Data

For this demo, we'll create synthetic binary fingerprints. In practice, you'd use molecular fingerprints, text features, or any binary/sparse data.

We'll create 3 clusters with different feature patterns so the tree shows clear structure.

In [2]:
np.random.seed(42)

n_samples = 30000
n_bits = 2048
n_clusters = 3

# Create clustered data - each cluster has different "active" bit ranges
fingerprints = np.zeros((n_samples, n_bits), dtype=np.uint8)
cluster_labels = []

for i in range(n_samples):
    cluster = i % n_clusters
    cluster_labels.append(f"Cluster_{cluster}")

    # Base features for this cluster (creates similarity within cluster)
    base_start = cluster * 500
    base_bits = np.random.choice(400, 50, replace=False) + base_start
    fingerprints[i, base_bits] = 1

    # Some random bits (adds variation)
    random_bits = np.random.choice(n_bits, 20, replace=False)
    fingerprints[i, random_bits] = 1

# Also create some continuous values for coloring
continuous_values = np.random.uniform(0, 100, n_samples)

print(f"Created {n_samples} fingerprints with {n_bits} bits each")
print(f"Average bits set: {fingerprints.sum(axis=1).mean():.1f}")
print(f"Clusters: {set(cluster_labels)}")


Created 30000 fingerprints with 2048 bits each
Average bits set: 69.5
Clusters: {'Cluster_0', 'Cluster_1', 'Cluster_2'}


## Step 1: MinHash Encoding

MinHash compresses your fingerprints into compact signatures while preserving Jaccard similarity.

**Key parameter:** `num_perm` controls the signature size (more = higher accuracy but slower)

In [3]:
# Create MinHash encoder
mh = MinHash(num_perm=128, seed=42)

# Encode fingerprints to signatures
# Use batch_from_binary_array for best performance
signatures = mh.batch_from_binary_array(fingerprints)

print(f"Original shape: {fingerprints.shape}")
print(f"Signature shape: {signatures.shape}")
print(f"Compression ratio: {fingerprints.nbytes / signatures.nbytes:.1f}x")


Original shape: (30000, 2048)
Signature shape: (30000, 128)
Compression ratio: 2.0x


## Step 2: Build LSH Forest Index

The LSH Forest enables fast approximate nearest neighbor queries.

**Key parameters:**
- `d`: Signature dimension (must match MinHash `num_perm`)
- `l`: Number of prefix trees (more = better recall, more memory)

In [4]:
# Create LSH Forest
lsh = LSHForest(d=128, l=64)

# Add all signatures
lsh.batch_add(signatures)

# Build the index (required before queries)
lsh.index()

print(f"Indexed {lsh.size} signatures")
print(f"Index is ready: {lsh.is_indexed}")


Indexed 30000 signatures
Index is ready: True


## Step 3: Compute Layout

This is where the magic happens! The layout algorithm:
1. Builds a k-nearest neighbor graph
2. Extracts the minimum spanning tree (MST)
3. Applies force-directed layout

**Key parameters:**
- `k`: Neighbors per point (higher = better connectivity)
- `kc`: Search quality multiplier (higher = better neighbors)
- `node_size`: Controls spacing (higher = more spread out)

In [5]:
# Configure layout
cfg = LayoutConfig()
cfg.k = 20          # 20 nearest neighbors
cfg.kc = 50         # Search 20*50 = 1000 candidates
cfg.node_size = 1/30  # Medium spread
cfg.fme_iterations = 500  # Layout iterations
cfg.deterministic = True  # Reproducible results
cfg.seed = 42

# Compute layout
x, y, s, t = layout_from_lsh_forest(lsh, cfg)

print("Layout computed!")
print(f"  Nodes: {len(x)}")
print(f"  Edges: {len(s)} (MST has n-1 edges for connected tree)")
print(f"  Coordinate range: x=[{x.min():.2f}, {x.max():.2f}], y=[{y.min():.2f}, {y.max():.2f}]")


Layout computed!
  Nodes: 30000
  Edges: 29999 (MST has n-1 edges for connected tree)
  Coordinate range: x=[-0.50, 0.50], y=[-0.50, 0.50]


## Step 4: Create Visualization

Now let's create an interactive HTML visualization with:
- Categorical coloring by cluster
- Continuous coloring by value
- Tree edges from layout output
- Labels in tooltips

In [6]:
# Create visualizer
viz = TmapViz()
viz.title = "TMAP Quickstart Demo"
viz.background_color = "#FFFFFF"  # White background
viz.point_size = 4.0
viz.opacity = 0.85

# Set coordinates
viz.set_points(x, y)
viz.set_edges(s, t)
viz.set_edge_style(color="#111111", width=1.5, opacity=0.35)

# Add categorical coloring (clusters)
viz.add_color_layout(
    name="Cluster",
    values=cluster_labels,
    categorical=True,
    color="Set1"  # Colormap for categories
)

# Add continuous coloring (values)
viz.add_color_layout(
    name="Value",
    values=continuous_values.tolist(),
    categorical=False,
    color="viridis"  # Colormap for continuous
)

# Add labels for tooltips
viz.add_label("ID", [f"Point_{i}" for i in range(n_samples)])

print(f"Visualization configured with {viz.n_points} points")
print(f"Color layouts: {[l.name for l in viz.layouts]}")
print(f"Labels: {[l.name for l in viz.labels]}")


Visualization configured with 30000 points
Color layouts: ['Cluster', 'Value']
Labels: ['Cluster', 'Value', 'ID']


## Step 5: Save and View
Before saving we can visualize the TMAP. It is interactive (try hovering over points to see the labels) and allows for very useful features like Lasso selection

In [7]:
widget = viz.to_widget()
widget.show()


/var/folders/zw/c0mr_7jx7t3bz1k6_9n9wh3r0000gn/T/ipykernel_10334/2819450684.py:1: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  widget = viz.to_widget()


We can select points using the mouse. Long press the left click and then simply drag to select your points. Then, once selected (you should see them highlighted, do `widget.selection()` to get the index)

In [15]:
widget.selection()


array([1], dtype=uint32)

It pretty much works like a pandas dataframe, we can also pass the index to select it in the map


In [16]:
widget = viz.to_widget()
widget.selection([1])
widget.show()


/var/folders/zw/c0mr_7jx7t3bz1k6_9n9wh3r0000gn/T/ipykernel_10334/1971226951.py:1: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  widget = viz.to_widget()


Ups... That's a bit tricky to find.
If you have trouble finding the molecule selected you can zoom in!


In [17]:
widget = viz.to_widget()
widget.selection([1])
widget.show()


/var/folders/zw/c0mr_7jx7t3bz1k6_9n9wh3r0000gn/T/ipykernel_10334/1971226951.py:1: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  widget = viz.to_widget()


In [18]:
widget.zoom(to=widget.selection())


Finally, if that seems like a good TMAP, you can save the visualization to an HTML file. You can open it in any web browser!


In [13]:
# Save to HTML
output_path = viz.write_html("./")

print(f"\nVisualization saved to: {output_path}")
print("\nOpen in your browser and try:")
print("  - Pan: Click and drag")
print("  - Zoom: Mouse wheel")
print("  - Hover: See point details")
print("  - Dropdown: Change color scheme")



Visualization saved to: TMAP Quickstart Demo.html

Open in your browser and try:
  - Pan: Click and drag
  - Zoom: Mouse wheel
  - Hover: See point details
  - Dropdown: Change color scheme


A much more detailed tutorial and example notebook is available in the [jscatter demo notebook](04_jscatter_demo.ipynb)

## Summary

You've learned the complete TMAP pipeline:

```python
# 1. Encode
mh = MinHash(num_perm=128, seed=42)
signatures = mh.batch_from_binary_array(fingerprints)

# 2. Index
lsh = LSHForest(d=128, l=64)
lsh.batch_add(signatures)
lsh.index()

# 3. Layout
cfg = LayoutConfig()
cfg.k = 20
cfg.kc = 50
x, y, s, t = layout_from_lsh_forest(lsh, cfg)

# 4. Visualize
viz = TmapViz()
viz.set_points(x, y)
viz.set_edges(s, t)
viz.set_edge_style(color="#111111", width=1.5, opacity=0.35)
viz.add_color_layout("name", values, categorical=False)
viz.write_html("./")
```

## Next Steps

- See `02_minhash_deep_dive.ipynb` for encoding details
- See `03_tuning_parameters.ipynb` for parameter optimization
- See `04_jscatter_demo.ipynb` for visualization
- Check the [documentation](../docs/) for full API reference